# Plot the UltraBench Tasks

Plot the class distribution of every classification task and one example image, with its
segmentation mask overlaid, for every segmentation task.

In [ ]:
from collections import Counter
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

TASK_CONFIG_DIR = Path("../src/ultrabench/training/config/task")

TASK_DISPLAY_NAMES = {
    # Classification
    "aul_mass_classification": "AUL Mass",
    "busbra_histology_classification": "BUS-BRA Hist.",
    "busbra_pathology_classification": "BUS-BRA Path.",
    "camus_classification": "CAMUS",
    "covid_blues_classification": "COVID-BLUES",
    "fetal_planes_brain_plane_classification": "Fetal Brain",
    "fetal_planes_plane_classification": "Fetal Plane",
    "gbcu_classification": "GBCU",
    "masld_classification": "MASLD",
    "mmotu_classification": "MMOTU",
    "tn3k_classification": "TN3K",
    # Segmentation
    "aul_liver_segmentation": "AUL Liver",
    "aul_mass_segmentation": "AUL Mass",
    "busbra_segmentation": "BUS-BRA",
    "camus_segmentation": "CAMUS",
    "mmotu_segmentation": "MMOTU",
    "open_kidney_capsule_segmentation": "OK Capsule",
    "open_kidney_regions_segmentation": "OK Regions",
    "psfhs_segmentation": "PSFHS",
    "stmus_segmentation": "STMUS",
    "tn3k_segmentation": "TN3K",
}
PATHO_CLS_TASKS = [
    "aul_mass_classification",
    "busbra_histology_classification",
    "busbra_pathology_classification",
    "covid_blues_classification",
    "gbcu_classification",
    "masld_classification",
    "mmotu_classification",
    "tn3k_classification",
]
NONPATHO_CLS_TASKS = [
    "camus_classification",
    "fetal_planes_brain_plane_classification",
    "fetal_planes_plane_classification",
]
PATHO_SEG_TASKS = [
    "aul_mass_segmentation",
    "busbra_segmentation",
    "mmotu_segmentation",
    "tn3k_segmentation",
]
ANATOMICAL_SEG_TASKS = [
    "aul_liver_segmentation",
    "camus_segmentation",
    "open_kidney_capsule_segmentation",
    "open_kidney_regions_segmentation",
    "psfhs_segmentation",
    "stmus_segmentation",
]

CLASSIFICATION_TASKS = PATHO_CLS_TASKS + NONPATHO_CLS_TASKS
SEGMENTATION_TASKS = PATHO_SEG_TASKS + ANATOMICAL_SEG_TASKS

# Abbreviations for the class names that are too long to fit on the X-axis of
# a subplot. Otherwise use `class_label` names.
CLASS_DISPLAY_NAMES = {
    # BUS-BRA histology
    "invasive ductal carcinoma": "IDC",
    "invasive lobular carcinoma": "ILC",
    "fibrocystic changes": "Fibrocystic",
    "intraductal papilloma": "Papilloma",
    "sclerosing adenosis": "Sclerosing aden.",
    # MMOTU tumour type
    "serous_cystadenoma": "Serous cyst.",
    "mucinous_cystadenoma": "Mucinous cyst.",
    "high_grade_serous_cystadenocarcinoma": "High grade serous",
    # Fetal planes
    "Fetal abdomen": "Abdomen",
    "Fetal brain": "Brain",
    "Fetal femur": "Femur",
    "Fetal thorax": "Thorax",
    "Maternal cervix": "Cervix",
    "Trans-thalamic": "Thalamic",
    "Trans-cerebellum": "Cerebellum",
    "Trans-ventricular": "Ventricular",
}

# The structures encoded by each segmentation mask, in class-index order
# starting from 1 (index 0 is always the background). Taken from the dataset
# cards in `docs/dataset_cards/`.
MASK_CLASS_NAMES = {
    "aul_liver_segmentation": ["Liver"],
    "aul_mass_segmentation": ["Mass"],
    "busbra_segmentation": ["Tumour"],
    "camus_segmentation": ["LV endocardium", "LV myocardium", "LA"],
    "mmotu_segmentation": ["Tumour"],
    "open_kidney_capsule_segmentation": ["Capsule"],
    "open_kidney_regions_segmentation": ["Central echo cx.", "Medulla", "Cortex"],
    "psfhs_segmentation": ["Pubic symphysis", "Fetal head"],
    "stmus_segmentation": ["Muscle"],
    "tn3k_segmentation": ["Nodule"],
}

# The row of the test split to take each segmentation example from. Increase a
# value to hand-pick a more representative image for that task.
SEGMENTATION_EXAMPLE_INDICES = {task: 0 for task in SEGMENTATION_TASKS}

# Configure plotting libraries
plt.rcParams["figure.figsize"] = (6.5, 2.48)
sns.set_theme(style="whitegrid", context="paper", palette="colorblind", font_scale=0.8)

# Font scale used for the multi-panel grids, small enough to keep the rotated
# class names and the per-panel legends legible at this panel size.
TASK_FONT_SCALE = 0.6

# One colour per foreground mask class, so that class 1 is the same colour in
# every panel of the segmentation figure.
MASK_COLORS = sns.color_palette("colorblind", n_colors=3)

# The splits to compare in the class distribution figure, and the source of
# the segmentation examples. `test.256` holds the same rows as `test`, already
# resized and zero-padded to 256x256, so the example panels line up without
# any further resizing.
SPLITS = {"train": "train", "test": "test.256"}

## Load the Task Registry

Read the Hydra task configs to get the Hugging Face repository, subset, revision and
target column of every task.

In [ ]:
def load_task_configs(config_dir: Path) -> dict[str, dict]:
    """Load the Hydra task configs of the active benchmark tasks.

    Args:
        config_dir: The directory holding one YAML config per task.

    Returns:
        A dict mapping each task name to its parsed config.

    Raises:
        FileNotFoundError: If the config directory does not exist.
    """
    if not config_dir.is_dir():
        raise FileNotFoundError(f"No task config directory at '{config_dir}'.")

    return {
        path.stem: yaml.safe_load(path.read_text())
        for path in sorted(config_dir.glob("*.yaml"))
    }


task_configs = load_task_configs(TASK_CONFIG_DIR)

SUMMARY_KEYS = ["dataset", "subset", "revision", "type", "target", "num_classes"]

pd.DataFrame(
    [
        {"task": task, **{key: config[key] for key in SUMMARY_KEYS}}
        for task, config in task_configs.items()
    ]
)

## Count the Classes of Each Classification Task

Count the images per class in the `train` and `test` splits of every classification task.

The first run downloads every dataset that is not already in the Hugging Face cache
(`HF_HOME`), which takes tens of gigabytes and a long time. Later runs read the cache.

Every published `train` split holds at most 1,000 examples, and the surplus is either moved
into `test` or discarded, so the two bars of a class are not comparable in size. See
`docs/dataset_format.md` for the per-dataset rule.

In [ ]:
def class_label(name: str) -> str:
    """Return the display label of a class name.

    Args:
        name: The raw class name taken from the dataset features.

    Returns:
        The abbreviation from `CLASS_DISPLAY_NAMES`, or the raw name with
        underscores replaced by spaces and the first letter capitalized.
    """
    if name in CLASS_DISPLAY_NAMES:
        return CLASS_DISPLAY_NAMES[name]

    return name.replace("_", " ").capitalize()


def count_classes(config: dict) -> pd.DataFrame:
    """Count the images per class and split of a classification task.

    Only the target column of each split is materialized, so the images are
    never decoded.

    Args:
        config: The Hydra config of the task.

    Returns:
        A long form data frame with one row per image and `class` and `split`
        columns, ordered so that `class` is a categorical in class-index
        order.
    """
    rows = []
    class_names = None
    for split_label, split in SPLITS.items():
        dataset = datasets.load_dataset(
            config["dataset"],
            name=config["subset"],
            revision=config["revision"],
            split=split,
        )
        feature = dataset.features[config["target"]]
        class_names = [class_label(name) for name in feature.names]

        counts = Counter(dataset[config["target"]])
        for index, name in enumerate(class_names):
            rows.extend([(name, split_label)] * counts[index])

    counts_df = pd.DataFrame(rows, columns=["class", "split"])
    counts_df["class"] = pd.Categorical(
        counts_df["class"], categories=class_names, ordered=True
    )

    return counts_df


class_distributions = {
    task: count_classes(task_configs[task]) for task in CLASSIFICATION_TASKS
}

pd.DataFrame(
    [
        {
            "task": task,
            "classes": counts_df["class"].nunique(),
            "train": int((counts_df["split"] == "train").sum()),
            "test": int((counts_df["split"] == "test").sum()),
        }
        for task, counts_df in class_distributions.items()
    ]
)

## Plot the Class Distributions

In [ ]:
def plot_class_distribution(ax, task: str, plot_df: pd.DataFrame) -> None:
    """Plot the per-class train and test image counts of one task.

    Args:
        ax: The axes to draw the panel on.
        task: The name of the task to plot.
        plot_df: A long form data frame with `class` and `split` columns.

    Returns:
        None.
    """
    sns.countplot(
        data=plot_df,
        x="class",
        hue="split",
        hue_order=list(SPLITS),
        ax=ax,
        legend=False,
    )

    ax.set_title(TASK_DISPLAY_NAMES[task])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_box_aspect(1.0)


with sns.plotting_context("paper", font_scale=TASK_FONT_SCALE):
    fig, axes = plt.subplots(3, 4, figsize=(6.5, 6))

    for ax, task in zip(axes.flat, CLASSIFICATION_TASKS):
        plot_class_distribution(ax, task, class_distributions[task])

    for ax in axes.flat[len(CLASSIFICATION_TASKS) :]:
        ax.set_axis_off()

    for ax in axes[:, 0]:
        ax.set_ylabel("Images")

    # Put the shared split legend where the unused panel would have been.
    colors = sns.color_palette("colorblind", n_colors=len(SPLITS))
    axes.flat[0].legend(
        handles=[
            Patch(facecolor=color, label=split.capitalize())
            for split, color in zip(SPLITS, colors)
        ],
        loc="upper left",
        title="Split",
        # frameon=False,
    )

    plt.tight_layout()
    plt.savefig("classification_task_class_distributions.pdf")
    plt.savefig("classification_task_class_distributions.png", dpi=300)
    plt.show()

## Load an Example of Each Segmentation Task

Take one example per task from the `test.256` split, so that every panel is already a
256x256 image and the grid lines up without resizing.

In [ ]:
MAX_EXAMPLE_SCAN = 200


def load_example(config: dict, start_index: int) -> tuple[np.ndarray, np.ndarray]:
    """Load one example of a segmentation task with a non-empty mask.

    Scans forward from `start_index` one row at a time, so that only the
    images of the inspected rows are decoded.

    Args:
        config: The Hydra config of the task.
        start_index: The row of the test split to start scanning from.

    Returns:
        The RGB image and the single-channel, multi-class mask of the first
        example at or after `start_index` with a non-empty mask.

    Raises:
        ValueError: If no example with a non-empty mask is found within
            `MAX_EXAMPLE_SCAN` rows of `start_index`.
    """
    dataset = datasets.load_dataset(
        config["dataset"],
        name=config["subset"],
        revision=config["revision"],
        split=SPLITS["test"],
    )

    stop = min(start_index + MAX_EXAMPLE_SCAN, len(dataset))
    for index in range(start_index, stop):
        row = dataset[index]
        if row[config["target"]] is None:
            continue

        mask = np.array(row[config["target"]])
        if mask.any():
            return np.array(row["image"].convert("RGB")), mask

    raise ValueError(
        f"No example with a non-empty '{config['target']}' mask in rows "
        f"{start_index}-{stop} of '{config['dataset']}'."
    )


segmentation_examples = {
    task: load_example(task_configs[task], SEGMENTATION_EXAMPLE_INDICES[task])
    for task in SEGMENTATION_TASKS
}

pd.DataFrame(
    [
        {
            "task": task,
            "image": str(image.shape),
            "mask_classes": np.unique(mask).tolist(),
        }
        for task, (image, mask) in segmentation_examples.items()
    ]
)

## Plot the Segmentation Examples

In [ ]:
MASK_ALPHA = 0.45


def plot_segmentation_example(
    ax, task: str, image: np.ndarray, mask: np.ndarray
) -> None:
    """Plot one segmentation example with its mask overlaid.

    Each foreground class is drawn as a translucent patch with a solid
    outline, so that the underlying ultrasound stays visible.

    Args:
        ax: The axes to draw the panel on.
        task: The name of the task to plot.
        image: The RGB ultrasound image.
        mask: The single-channel, multi-class segmentation mask.

    Returns:
        None.
    """
    ax.imshow(image)

    class_names = MASK_CLASS_NAMES[task]
    overlay = np.zeros((*mask.shape, 4))
    for index, color in enumerate(MASK_COLORS[: len(class_names)]):
        foreground = (mask == index + 1).astype(float)
        overlay[foreground == 1.0] = (*color, MASK_ALPHA)
        ax.contour(foreground, levels=[0.5], colors=[color], linewidths=0.5)

    ax.imshow(overlay, interpolation="nearest")

    ax.set_title(TASK_DISPLAY_NAMES[task])
    ax.set_axis_off()
    ax.legend(
        handles=[
            Patch(facecolor=MASK_COLORS[index], alpha=MASK_ALPHA, label=name)
            for index, name in enumerate(class_names)
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.0),
        ncol=1,
        frameon=False,
        handlelength=1.0,
        borderpad=0.0,
    )


with sns.axes_style("white"), sns.plotting_context("paper", font_scale=0.5):
    fig, axes = plt.subplots(2, 5, figsize=(6.5, 3.6))

    for ax, task in zip(axes.flat, SEGMENTATION_TASKS):
        plot_segmentation_example(ax, task, *segmentation_examples[task])

    plt.tight_layout()
    plt.savefig("segmentation_task_examples.pdf")
    plt.savefig("segmentation_task_examples.png", dpi=300)
    plt.show()